In [ ]:
#!/usr/bin/env python3
import re, gzip
from pathlib import Path
import pandas as pd

# ========= Inputs =========
tss_file = "protein_coding_TSS.tsv"
tts_file = "protein_coding_TTS.tsv"

gff_v48 = "/rhome/zli529/lab/PlasmoDB_Genome/PlasmoDB_v48/PlasmoDB-48_Pfalciparum3D7.gff"
gff_v68 = "/rhome/zli529/lab/PlasmoDB_Genome/PlasmnDB_v68/PlasmoDB-68_Pfalciparum3D7.gff"

out_file = "combined_TSS_TTS_with_signed_distances.tsv"
# ==========================

PF_RE = re.compile(r"(PF3D7_\d{7})")

def open_text(path: str):
    return gzip.open(path, "rt") if str(path).endswith(".gz") else open(path, "rt")

def extract_pf_id(text: str):
    m = PF_RE.search(text or "")
    return m.group(1) if m else None

def safe_num(x):
    try:
        if pd.isna(x): return None
        return float(x)
    except Exception:
        return None

def parse_v48_gene_sites(gff_path: str) -> dict:
    """Parse v48 'gene' features -> gene_id: {chr,strand,tss,tts}."""
    annot = {}
    with open_text(gff_path) as fh:
        for line in fh:
            if not line or line.startswith("#"): continue
            p = line.rstrip("\n").split("\t")
            if len(p) < 9: continue
            chrom, src, feat, start, end, score, strand, phase, attrs = p
            if feat != "gene": continue
            gid = extract_pf_id(attrs) or (extract_pf_id((re.search(r"(?:ID|Name|gene_id|locus_tag)=([^;]+)", attrs) or [None]).group(1)) if re.search(r"(?:ID|Name|gene_id|locus_tag)=([^;]+)", attrs) else None)
            if not gid: continue
            try:
                s, e = int(start), int(end)
            except ValueError:
                continue
            tss = s if strand == "+" else e
            tts = e if strand == "+" else s
            annot[gid] = {"chr": chrom, "strand": strand, "tss": tss, "tts": tts}
    return annot

def parse_v68_gene_sites_by_id(gff_path: str, keep_ids: set) -> dict:
    """Parse v68 by PF3D7 ID only (ignore feature type) -> gene_id: {chr,strand,tss,tts}."""
    ann = {}
    with open_text(gff_path) as fh:
        for line in fh:
            if not line or line.startswith("#"): continue
            p = line.rstrip("\n").split("\t")
            if len(p) < 9: continue
            chrom, src, feat, start, end, score, strand, phase, attrs = p
            gid = extract_pf_id(attrs)
            if not gid or gid not in keep_ids: continue
            try:
                s, e = int(start), int(end)
            except ValueError:
                continue
            tss = s if strand == "+" else e
            tts = e if strand == "+" else s
            ann[gid] = {"chr": chrom, "strand": strand, "tss": tss, "tts": tts}
    return ann

def main():
    # --- Load predictions ---
    tss = pd.read_csv(tss_file, sep="\t")
    tts = pd.read_csv(tts_file, sep="\t")

    # Normalize IDs (drop isoform suffixes like .1, .2)
    tss["gene_id"] = tss["gene_id"].str.replace(r"\.\d+$", "", regex=True)
    tts["gene_id"] = tts["gene_id"].str.replace(r"\.\d+$", "", regex=True)

    # Keep valid PF3D7 IDs
    tss = tss[tss["gene_id"].str.match(PF_RE, na=False)].copy()
    tts = tts[tts["gene_id"].str.match(PF_RE, na=False)].copy()

    # Ensure numeric (keep floats to allow NaN)
    tss["tss_pos"] = pd.to_numeric(tss["tss_pos"], errors="coerce")
    tts["tts_pos"] = pd.to_numeric(tts["tts_pos"], errors="coerce")

    # Merge predictions
    tss = tss.drop_duplicates(subset=["chr","strand","gene_id"])
    tts = tts.drop_duplicates(subset=["chr","strand","gene_id"])
    pred = pd.merge(
        tss[["chr","strand","gene_id","tss_pos"]],
        tts[["chr","strand","gene_id","tts_pos"]],
        on=["chr","strand","gene_id"], how="inner"
    ).rename(columns={"tss_pos":"pred_tss","tts_pos":"pred_tts"})

    # --- Parse annotations ---
    ann48 = parse_v48_gene_sites(gff_v48)
    v48_ids = set(ann48.keys())
    pred = pred[pred["gene_id"].isin(v48_ids)].copy()  # align to v48 gene set
    ann68 = parse_v68_gene_sites_by_id(gff_v68, v48_ids)

    # Map annotated coords (original annotations)
    pred["v48_tss"] = pred["gene_id"].map(lambda g: ann48.get(g, {}).get("tss"))
    pred["v48_tts"] = pred["gene_id"].map(lambda g: ann48.get(g, {}).get("tts"))
    pred["v68_tss"] = pred["gene_id"].map(lambda g: ann68.get(g, {}).get("tss"))
    pred["v68_tts"] = pred["gene_id"].map(lambda g: ann68.get(g, {}).get("tts"))

    # --- Your mapping for output start/end ---
    # + strand: tss=start, tts=end ; - strand: tts=start, tss=end
    plus  = pred["strand"] == "+"
    minus = pred["strand"] == "-"

    pred.loc[plus,  "start"] = pred.loc[plus,  "pred_tss"]
    pred.loc[plus,  "end"]   = pred.loc[plus,  "pred_tts"]
    pred.loc[minus, "start"] = pred.loc[minus, "pred_tts"]
    pred.loc[minus, "end"]   = pred.loc[minus, "pred_tss"]

    # --- Signed changes per your exact rules ---
    def signed_abs(a, b, sign_fn):
        A, B = safe_num(a), safe_num(b)
        if A is None or B is None: return pd.NA
        mag = abs(A - B)
        sgn = sign_fn(A, B)
        if sgn == 0: return 0
        return (+mag) if sgn > 0 else (-mag)

    # strand '-' rules:
    def sign_minus_tss_vs_vtts(tss, vtts):  # for dist_to_v*_tts
        if tss > vtts: return +1
        if tss < vtts: return -1
        return 0
    def sign_minus_tss_vs_vtss(tss, vtss):  # for dist_to_v*_tss
        if tss < vtss: return +1
        if tss > vtss: return -1
        return 0

    # strand '+' rules:
    def sign_plus_tss_vs_vtss(tss, vtss):   # for dist_to_v*_tss
        if tss < vtss: return +1
        if tss > vtss: return -1
        return 0
    def sign_plus_tts_vs_vtts(tts, vtts):   # for dist_to_v*_tts
        if tts > vtts: return +1
        if tts < vtts: return -1
        return 0

    # Initialize output columns
    for col in ["dist_to_v48_tss","dist_to_v68_tss","dist_to_v48_tts","dist_to_v68_tts"]:
        pred[col] = pd.NA

    # Compute per strand group
    # minus: tss=end, use minus rules
    pred.loc[minus, "dist_to_v48_tss"] = [
        signed_abs(tss, vtss, sign_minus_tss_vs_vtss)
        for tss, vtss in zip(pred.loc[minus, "pred_tss"], pred.loc[minus, "v48_tss"])
    ]
    pred.loc[minus, "dist_to_v68_tss"] = [
        signed_abs(tss, vtss, sign_minus_tss_vs_vtss)
        for tss, vtss in zip(pred.loc[minus, "pred_tss"], pred.loc[minus, "v68_tss"])
    ]
    pred.loc[minus, "dist_to_v48_tts"] = [
        signed_abs(tss, vtts, sign_minus_tss_vs_vtts)
        for tss, vtts in zip(pred.loc[minus, "pred_tss"], pred.loc[minus, "v48_tts"])
    ]
    pred.loc[minus, "dist_to_v68_tts"] = [
        signed_abs(tss, vtts, sign_minus_tss_vs_vtts)
        for tss, vtts in zip(pred.loc[minus, "pred_tss"], pred.loc[minus, "v68_tts"])
    ]

    # plus: tss=start, tts=end, use plus rules
    pred.loc[plus, "dist_to_v48_tss"] = [
        signed_abs(tss, vtss, sign_plus_tss_vs_vtss)
        for tss, vtss in zip(pred.loc[plus, "pred_tss"], pred.loc[plus, "v48_tss"])
    ]
    pred.loc[plus, "dist_to_v68_tss"] = [
        signed_abs(tss, vtss, sign_plus_tss_vs_vtss)
        for tss, vtss in zip(pred.loc[plus, "pred_tss"], pred.loc[plus, "v68_tss"])
    ]
    pred.loc[plus, "dist_to_v48_tts"] = [
        signed_abs(tts, vtts, sign_plus_tts_vs_vtts)
        for tts, vtts in zip(pred.loc[plus, "pred_tts"], pred.loc[plus, "v48_tts"])
    ]
    pred.loc[plus, "dist_to_v68_tts"] = [
        signed_abs(tts, vtts, sign_plus_tts_vs_vtts)
        for tts, vtts in zip(pred.loc[plus, "pred_tts"], pred.loc[plus, "v68_tts"])
    ]

    # --- Arrange output (with original annotations too) ---
    out_cols = [
        "chr","start","end","strand","gene_id",
        "dist_to_v48_tss","dist_to_v68_tss","dist_to_v48_tts","dist_to_v68_tts",
        "v48_tss","v48_tts","v68_tss","v68_tts",
        "pred_tss","pred_tts"
    ]
    pred[out_cols].to_csv(out_file, sep="\t", index=False)

    print("\n[summary]")
    print("rows:", len(pred))
    for c in ["dist_to_v48_tss","dist_to_v68_tss","dist_to_v48_tts","dist_to_v68_tts"]:
        print(f"null {c}:", int(pd.isna(pred[c]).sum()))
    print(f"Saved -> {out_file}")

if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
import re
import gzip
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ==== Inputs (edit if needed) ====
predicted_file = "predicted_gene_boundaries.tsv"  # columns: chr, start, end, strand, gene_id
gff_v48 = "/rhome/zli529/lab/PlasmoDB_Genome/PlasmoDB_v48/PlasmoDB-48_Pfalciparum3D7.gff"
gff_v68 = "/rhome/zli529/lab/PlasmoDB_Genome/PlasmnDB_v68/PlasmoDB-68_Pfalciparum3D7.gff"
# =================================

OUT_KDE  = "gene_length_distribution_0_10000_kde.png"
OUT_HIST = "gene_length_distribution_0_10000_hist.png"

PF_RE = re.compile(r"(PF3D7_\d{7})")

def open_text(path: str):
    return gzip.open(path, "rt") if str(path).endswith(".gz") else open(path, "rt")

def parse_gff_gene_lengths(path: str) -> pd.Series:
    """
    Return Series of gene lengths from a GFF.
    Accepts 'gene' and 'protein_coding_gene' features.
    Falls back to empty if none found.
    """
    lengths = []
    if not Path(path).exists():
        print(f"[warn] GFF not found: {path}")
        return pd.Series(dtype="float64")

    with open_text(path) as fh:
        for line in fh:
            if not line or line.startswith("#"):
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 9:
                continue
            chrom, src, feat, start, end, score, strand, phase, attrs = fields
            if feat not in {"gene", "protein_coding_gene"}:
                continue
            # Require a PF3D7 gene id somewhere in the attributes (keeps scope clean)
            if not PF_RE.search(attrs):
                continue
            try:
                s, e = int(start), int(end)
            except ValueError:
                continue
            lengths.append(abs(e - s) + 1)

    s = pd.Series(lengths, dtype="float64")
    print(f"[info] Parsed {len(s)} gene lengths from {Path(path).name}")
    return s

def load_predicted_lengths(path: str) -> pd.Series:
    """
    Read predicted boundaries TSV (chr, start, end, strand, gene_id),
    compute length = |end - start| + 1.
    """
    df = pd.read_csv(path, sep="\t")
    for col in ("start", "end"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    lengths = (df["end"] - df["start"]).abs() + 1
    print(f"[info] Parsed {lengths.notna().sum()} predicted gene lengths from {Path(path).name}")
    return lengths

def main():
    # Load lengths
    pred_len = load_predicted_lengths(predicted_file)
    v48_len  = parse_gff_gene_lengths(gff_v48)
    v68_len  = parse_gff_gene_lengths(gff_v68)

    # Keep only 0–10,000 bp
    def clamp_0_10k(s: pd.Series) -> pd.Series:
        s = s.dropna()
        return s[(s >= 0) & (s <= 10000)]

    pred10 = clamp_0_10k(pred_len)
    v4810  = clamp_0_10k(v48_len)
    v6810  = clamp_0_10k(v68_len)

    # Pack into DataFrame for convenience
    df = pd.DataFrame({"Predicted": pred10, "v48": v4810, "v68": v6810})

    # ---- KDE plot (0–10,000) ----
    plt.figure(figsize=(8, 6))
    for col in df.columns:
        df[col].dropna().plot.kde(label=col)
    plt.xlabel("Gene length (bp)")
    plt.ylabel("Density")
    plt.title("Gene Length Distribution")
    plt.xlim(0, 10000)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_KDE, dpi=300)
    print(f"[saved] {OUT_KDE}")

    # ---- Histogram (normalized) ----
    plt.figure(figsize=(8, 6))
    bins = range(0, 10001, 200)  # 200-bp bins (adjust as you like)
    plt.hist([pred10, v4810, v6810],
             bins=bins, density=True, alpha=0.5, label=["Predicted", "v48", "v68"])
    plt.xlabel("Gene length (bp)")
    plt.ylabel("Density")
    plt.title("Gene Length Distribution")
    plt.xlim(0, 10000)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_HIST, dpi=300)
    print(f"[saved] {OUT_HIST}")

    # Optional: quick summary stats
    summ = pd.DataFrame({
        "count": [pred10.size, v4810.size, v6810.size],
        "median": [pred10.median(), v4810.median(), v6810.median()],
        "mean": [pred10.mean(), v4810.mean(), v6810.mean()]
    }, index=["Predicted", "v48", "v68"])
    print("\n[summary stats 0–10,000 bp]")
    print(summ.round(1).to_string())

if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
import pandas as pd

# Input / Output
tsv_file = "predicted_gene_boundaries.tsv"   # chr, start, end, strand, gene_id
bed_file = "predicted_gene_boundaries.bed"

def main():
    df = pd.read_csv(tsv_file, sep="\t")

    # BED format is 0-based start, so subtract 1
    df["bed_start"] = (df["start"].astype(int) - 1)
    df["bed_end"]   = df["end"].astype(int)

    # Build BED columns (chr, start, end, name, score, strand)
    bed = pd.DataFrame({
        "chr": df["chr"],
        "start": df["bed_start"],
        "end": df["bed_end"],
        "name": df["gene_id"],
        "score": 0,
        "strand": df["strand"]
    })

    # Save as BED (no header, tab-separated, integer coords)
    bed.to_csv(bed_file, sep="\t", header=False, index=False)
    print(f"Saved -> {bed_file}")

if __name__ == "__main__":
    main()
